# FleetRoute — International Routing Workspace · Map Policy Fix v8.2

Run the two code cells in order. Keep cell 2 running while using the workspace.

1. **Prepare:** verifies the embedded release, installs only missing Python/runtime prerequisites, and prepares the authenticated workspace files. It deliberately does **not** block on the large native optimizer build or on `osmium-tool`.
2. **Open:** launches FleetRoute immediately, verifies the local/public workspace, and overlaps fingerprint-bound native compilation plus first-use road-extraction-tool preparation with the operator's region/PBF/depot/work-point setup.

### v8.1 system-hardening and efficiency
- regional PBF integrity is accumulated **during transfer**, with bounded stalled-read timeouts, exact snapshot-bound resume, and fail-closed atomic publication;
- interrupted same-snapshot transfers retain verified-prefix bytes in small streaming chunks so retries resume rather than unnecessarily restarting large country downloads;
- prepared PBF staging and graph caches use same-filesystem hard links where safe, avoiding unnecessary country/graph copies;
- cached operating graphs are checked before Osmium/PBF work, and uncached road filtering writes directly to OSM XML when supported, with a correctness-preserving compatibility fallback;
- exact road snapping uses a conservative geographic edge grid plus exact projected geometry, removing the former work-point × full-edge scan while preserving exact nearest-road semantics;
- immutable materialization/dataset identities are cached, repeated graph serialization is avoided, and large extraction scratch files are removed only after durable graph publication;
- native clustering/solver preparation is fingerprinted, cross-process locked, transactional, concurrent on available CPU capacity, and warmed in the background; early jobs join the same lock with visible progress rather than duplicate compilation;
- `osmium-tool` is likewise best-effort prewarmed in the background; prepared graphs remain usable without it, while first uncached extraction waits visibly on the same installation lock;
- route validation, protected incumbent behavior, source/settings identity checks, and atomic solution publication are unchanged;
- the audited routing optimizer source is unchanged.

The application package is integrity checked before launch. Do not edit the embedded payload unless you regenerate the release manifest and checksums.
### v8.2 map-provider correction
- map pages and tile images send an origin-only Referer, as required by OSM; login/token redirects still suppress referrers and remove tokens from URLs;
- all map surfaces use the canonical OSM HTTPS endpoint; attribution links to the OSM copyright page;
- the image security allowlist includes the existing Esri terrain endpoint;
- background tiles are an external best-effort service, separate from downloaded Geofabrik road data. No tile bulk download or proxy bypass is added.

After updating an existing runtime, stop cell 2, run cell 1, then run cell 2 and reload the workspace. Existing runtime data is retained.


In [ ]:
# @title 1/2 — Prepare FleetRoute Workspace
import base64
import hashlib
import importlib.metadata
import io
import json
import os
import shutil
import subprocess
import sys
import zipfile
from pathlib import Path, PurePosixPath

APP_DIR = Path("/content/fleetroute_app")
RUNTIME_DIR = Path("/content/fleetroute-runtime")
DATA_DIR = Path("/content/fleetroute-data")
ENV_CONFIG_PATH = Path("/content/fleetroute_env.json")
INSTALL_MARKER = APP_DIR / ".embedded_app.sha256"
EMBEDDED_APP_SHA256 = "6662ff0220bc099bd67bfc319fc1928b4ab2619385288d824ab741bb8fbedc74"
EXPECTED_RELEASE_MANIFEST_SHA256 = "c21fd08329ff5574facf4ee90e1936d91543113c6c7e066db613e5e6834c360c"
EXPECTED_SOLVER_SHA256 = "89619e5ed68f00e70f78d1633795db23becd02c158319d80dcc910b31c772e48"
EMBEDDED_APP_ZIP_B64 = """..."""

RUNTIME_DIR.mkdir(parents=True, exist_ok=True)
DATA_DIR.mkdir(parents=True, exist_ok=True)

def sha256_file(path):
    h = hashlib.sha256()
    with Path(path).open("rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

def _safe_rel_path(text):
    rel = PurePosixPath(str(text))
    return bool(rel.parts) and not rel.is_absolute() and ".." not in rel.parts

def verify_release_tree(root):
    root = Path(root)
    manifest_path = root / "RELEASE_MANIFEST.json"
    if not root.is_dir() or not manifest_path.is_file() or manifest_path.is_symlink():
        return False, "application manifest missing"
    if sha256_file(manifest_path) != EXPECTED_RELEASE_MANIFEST_SHA256:
        return False, "application manifest checksum mismatch"
    try:
        manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
    except Exception as exc:
        return False, f"application manifest unreadable: {exc}"
    files = manifest.get("files")
    if not isinstance(files, dict) or manifest.get("file_count") != len(files):
        return False, "application manifest file table invalid"
    if manifest.get("solver_source_sha256") != EXPECTED_SOLVER_SHA256:
        return False, "application manifest solver identity mismatch"

    expected = {"RELEASE_MANIFEST.json"}
    for rel_text, meta in files.items():
        if not _safe_rel_path(rel_text) or not isinstance(meta, dict):
            return False, f"unsafe manifest path: {rel_text}"
        expected.add(rel_text)
        path = root.joinpath(*PurePosixPath(rel_text).parts)
        if not path.is_file() or path.is_symlink():
            return False, f"application file missing or invalid: {rel_text}"
        try:
            size = int(meta.get("bytes", -1))
            digest = str(meta.get("sha256", ""))
        except Exception:
            return False, f"application metadata invalid: {rel_text}"
        if path.stat().st_size != size or sha256_file(path) != digest:
            return False, f"application file checksum mismatch: {rel_text}"

    # Python may create bytecode after launch; those cache files are the only
    # tolerated extras. Any other source-tree drift forces a clean re-extract.
    actual = set()
    for path in root.rglob("*"):
        if path.is_symlink():
            return False, f"unexpected symlink: {path.relative_to(root)}"
        if not path.is_file():
            continue
        rel = path.relative_to(root)
        if path == INSTALL_MARKER or "__pycache__" in rel.parts or path.suffix == ".pyc":
            continue
        actual.add(rel.as_posix())
    if actual != expected:
        return False, "application package contains missing or unexpected files"
    if sha256_file(root / "solver.cpp") != EXPECTED_SOLVER_SHA256:
        return False, "optimizer source identity mismatch"
    return True, "verified"

raw = base64.b64decode(EMBEDDED_APP_ZIP_B64, validate=True)
actual_zip_sha = hashlib.sha256(raw).hexdigest()
if actual_zip_sha != EMBEDDED_APP_SHA256:
    raise RuntimeError(f"Embedded application checksum mismatch: {actual_zip_sha}")

cache_ok = False
cache_reason = "not installed"
if APP_DIR.is_dir() and INSTALL_MARKER.is_file():
    marker_ok = INSTALL_MARKER.read_text(encoding="utf-8").strip() == EMBEDDED_APP_SHA256
    if marker_ok:
        cache_ok, cache_reason = verify_release_tree(APP_DIR)
    else:
        cache_reason = "application marker mismatch"

if cache_ok:
    print("Application files: cached package integrity confirmed.")
else:
    if APP_DIR.exists():
        print(f"Application files: cache refresh required ({cache_reason}).")
    stage = Path("/content/.fleetroute-app-stage")
    shutil.rmtree(stage, ignore_errors=True)
    stage.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(io.BytesIO(raw)) as z:
        bad = z.testzip()
        if bad:
            raise RuntimeError(f"Embedded ZIP failed integrity check at {bad}")
        for info in z.infolist():
            rel = PurePosixPath(info.filename)
            if rel.is_absolute() or ".." in rel.parts or not rel.parts or rel.parts[0] != "fleetroute_app":
                raise RuntimeError(f"Unsafe embedded path: {info.filename}")
            mode = (info.external_attr >> 16) & 0o170000
            if mode == 0o120000:
                raise RuntimeError(f"Symlinks are not allowed in the embedded app: {info.filename}")
            inner = rel.parts[1:]
            if not inner:
                continue
            target = stage.joinpath(*inner)
            if info.is_dir():
                target.mkdir(parents=True, exist_ok=True)
                continue
            target.parent.mkdir(parents=True, exist_ok=True)
            with z.open(info) as src, target.open("wb") as dst:
                shutil.copyfileobj(src, dst)
            file_mode = (info.external_attr >> 16) & 0o777
            if file_mode:
                target.chmod(file_mode)
    stage_ok, stage_reason = verify_release_tree(stage)
    if not stage_ok:
        shutil.rmtree(stage, ignore_errors=True)
        raise RuntimeError(f"Extracted application integrity check failed: {stage_reason}")
    shutil.rmtree(APP_DIR, ignore_errors=True)
    os.replace(stage, APP_DIR)
    INSTALL_MARKER.write_text(EMBEDDED_APP_SHA256 + "\n", encoding="utf-8")
    cache_ok, cache_reason = verify_release_tree(APP_DIR)
    if not cache_ok:
        raise RuntimeError(f"Installed application integrity check failed: {cache_reason}")
    print("Application files: package integrity confirmed and extracted.")

env_config = {
    "VRP_RUNTIME_DIR": str(RUNTIME_DIR),
    "VRP_DATABASE_PATH": str(RUNTIME_DIR / "fleetroute.sqlite3"),
    "VRP_OSM_DATA_DIR": str(DATA_DIR / "osm"),
    "VRP_REGION_CACHE_DIR": str(DATA_DIR / "regional-cache"),
    "VRP_SOLVER_BIN": str(RUNTIME_DIR / "solver"),
    "VRP_CLUSTER_BIN": str(RUNTIME_DIR / "cluster"),
    "VRP_APP_NAME": "FleetRoute Routing Workspace",
    "PYTHONUNBUFFERED": "1",
    "PYTHONDONTWRITEBYTECODE": "1",
}
for key in ("VRP_OSM_DATA_DIR", "VRP_REGION_CACHE_DIR"):
    Path(env_config[key]).mkdir(parents=True, exist_ok=True)
ENV_CONFIG_PATH.write_text(json.dumps(env_config, indent=2), encoding="utf-8")
os.environ.update(env_config)

requirements = APP_DIR / "requirements-colab.txt"
try:
    from packaging.requirements import Requirement
except Exception:
    from pip._vendor.packaging.requirements import Requirement

unmet = []
for line in requirements.read_text(encoding="utf-8").splitlines():
    line = line.strip()
    if not line or line.startswith("#"):
        continue
    req = Requirement(line)
    try:
        installed = importlib.metadata.version(req.name)
    except importlib.metadata.PackageNotFoundError:
        unmet.append(line)
        continue
    if req.specifier and installed not in req.specifier:
        unmet.append(line)

if unmet:
    print("Installing only missing/incompatible Python packages:", ", ".join(unmet))
    subprocess.check_call([
        sys.executable, "-m", "pip", "install",
        "--disable-pip-version-check", "--prefer-binary", "--no-input",
        *unmet,
    ])
    subprocess.check_call([sys.executable, "-c",
        "import fastapi,uvicorn,httpx,networkx,osmnx,shapely,pandas,numpy,scipy,sklearn,multipart,matplotlib"
    ])
else:
    print("Python dependencies: compatible environment detected; pip skipped.")

# g++ is normally already present in Google Colab. It is the only system tool
# required for the native optimizer itself; install it only when genuinely
# missing. Osmium is intentionally NOT a startup prerequisite in v8.1: Cell 2
# prewarms it in the background, and prepared road-graph caches can reopen
# without it.
if shutil.which("g++") is None:
    apt = shutil.which("apt-get")
    if not apt or (hasattr(os, "geteuid") and os.geteuid() != 0):
        raise RuntimeError("g++ is required to build FleetRoute's native optimizer but is not available on this host.")
    print("Native compiler missing; installing g++ (one-time prerequisite)...")
    subprocess.check_call([apt, "update", "-qq"])
    subprocess.check_call([apt, "install", "-y", "-qq", "g++"])
else:
    print("Native compiler: g++ available.")

# Do not block workspace startup on the very large -O3 native build or on
# osmium-tool. launch_colab.py starts both fingerprint-bound warmups in the
# background as Cell 2 opens the UI. Early optimize/extraction jobs join the
# same locks and display measured waiting progress; no duplicate build/install
# is started.
print("✓ FleetRoute application package is verified and ready.")
print("Run cell 2 now: the workspace opens while native engines and the first-use road extraction tool warm in the background when needed.")
print("App SHA-256:", EMBEDDED_APP_SHA256)
print("Application manifest SHA-256:", EXPECTED_RELEASE_MANIFEST_SHA256)


In [ ]:
# @title 2/2 — Open FleetRoute Workspace
import json
import os
import runpy
from pathlib import Path

APP_DIR = Path("/content/fleetroute_app")
ENV_CONFIG_PATH = Path("/content/fleetroute_env.json")
if not APP_DIR.is_dir() or not ENV_CONFIG_PATH.is_file():
    raise RuntimeError("Run cell 1 first.")

os.environ.update(json.loads(ENV_CONFIG_PATH.read_text(encoding="utf-8")))
previous = Path.cwd()
try:
    os.chdir(APP_DIR)
    runpy.run_path(str(APP_DIR / "launch_colab.py"), run_name="__main__")
finally:
    os.chdir(previous)
